In [ ]:
import lab.fullcontrol.infinaxis as fci
from math import sin, cos, tau


In [ ]:
EW = 0.6
EH = 0.3

print_settings = {'extrusion_width': EW,'extrusion_height': EH}
head_chain = [fci.Axis(name='B')]
bed_chain = [fci.Axis(name='C')]
Point = fci.configure_point(head_chain, bed_chain)

gcode_controls = fci.GcodeControls(
    head_chain = head_chain,
    bed_chain = bed_chain,
    initialization_data=print_settings 
    )

### QUESITON: I've created a simpler version of the next code cell in a new code cell immediately after it. Check this is okay, then delete the more complex code cell just below this comment. The intention is that readers don't need to think about the geometry/python stuff too much and focus more on the fci-specific aspects.

In [ ]:
def vase_from_trace(trace_points,density):
    steps = []
    for i in range(len(trace_points)-1):
        p = trace_points[i]
        p_next = trace_points[i+1]
        r = (p.x**2 + p.y**2)**0.5
        r_next = (p_next.x**2 + p_next.y**2)**0.5
        dr = r_next - r
        dz = p_next.z-p.z
        ptilt = p.b
        dtilt = p_next.b-ptilt
            
        dc = p_next.c-p.c
        for j in range(density):
            angle = p.c + dc * j / density
            tilt = ptilt + dtilt * j / density
            r_final = r + dr * j / density
            z_final = p.z + dz * j / density
   
            steps.append(Point(x=r_final*sin(angle/360*tau), y=r_final*cos(angle/360*tau), z=z_final, b=tilt, c=angle))
    
    return steps

density = 360
r_start = 10
r_tilt = 5 # 10 layers
tilt_start = 0
tilt_end = 90
h = EH # height of each layer
z_start = h*0.5 # starting z height
layers = 20 # number of layers in the z direction for first segment
arc_layers = int((r_tilt * (tilt_end - tilt_start) / 360 *tau)/h)
d_tilted = 5
layers_tilted = int(d_tilted/h)

trace = []

for i in range(layers):
    r = r_start
    tilt = tilt_start
    angle = i * 360
    z = z_start + h * i

    trace.append(Point(x=r, y=0, z=z, b=tilt, c=angle))

angle_offset = fci.last_point(trace).c
z_offset = fci.last_point(trace).z

for i in range(1,arc_layers):
    tilt = tilt_start + (tilt_end - tilt_start) * i / (arc_layers - 1)
    r = r_start+r_tilt*(1-cos(tilt*tau/360))
    z = z_offset+r_tilt*(sin(tilt*tau/360))
    angle = i * 360 + angle_offset
    trace.append(Point(x=r, y=0, z=z, b=-tilt, c=angle))

angle_offset = fci.last_point(trace).c
z_offset = fci.last_point(trace).z
r_offset = fci.last_point(trace).x

for i in range(1,layers_tilted):
    tilt = tilt_end
    r = r_offset + d_tilted * i / (layers_tilted - 1) * (sin(tilt*tau/360))
    z = z_offset + d_tilted * i / (layers_tilted - 1) * cos(tilt*tau/360)
    angle = i * 360 + angle_offset
    trace.append(Point(x=0, y=r, z=z, b=-tilt, c=angle))

steps = vase_from_trace(trace, density)
steps.append(fci.last_point(trace))

for step in steps:
    if isinstance(step, fci.Point):
        # color is a gradient from A=0 (blue) to A=90 (red)
        step.color = [((abs(step.b))/90), 0, 1-((abs(step.b))/90)]

fig = fci.transform(steps, 'fig', fci.PlotControls(color_type='manual',style='tube', zoom=0.75), show_tips=False)
fig.show()

gcode = fci.transform(steps,'gcode',gcode_controls)
print('first ten gcode lines:\n' + '\n'.join(gcode.split('\n')[:10]))
print('')
print('final ten gcode lines:\n' + '\n'.join(gcode.split('\n')[-10:]))

#### same style of five-axis path with simpler direct construction

This version uses one spiral turn per layer. `B` tilt increases by 1 degree per layer for 90 layers. The layer-to-layer spacing is kept constant by splitting the same step length into radial and vertical components with sine/cosine.


In [ ]:
# simpler direct version of the five-axis demo path

density = 360
layers = 90
layer_gap = EH
r = 10
z = EH / 2
steps = []

for layer in range(layers):
    tilt = layer
    dr = layer_gap * sin(tilt / 360 * tau)
    dz = layer_gap * cos(tilt / 360 * tau)
    for j in range(density):
        t = j / density
        angle = 360 * (layer + t)
        r_now = r + dr * t
        z_now = z + dz * t
        steps.append(Point(x=r_now * sin(angle / 360 * tau), y=r_now * cos(angle / 360 * tau), z=z_now, b=-tilt, c=angle))
    r += dr
    z += dz

for step in steps:
    step.color = [abs(step.b) / 90, 0, 1 - abs(step.b) / 90]

fig = fci.transform(steps, 'fig', fci.PlotControls(color_type='manual', style='tube', zoom=0.75), show_tips=False)
fig.show()

gcode = fci.transform(steps, 'gcode', gcode_controls)
print('first ten gcode lines:\n' + '\n'.join(gcode.split('\n')[:10]))
print('')
print('final ten gcode lines:\n' + '\n'.join(gcode.split('\n')[-10:]))
